# p1 · Term 추출 (Gemini)

문서(text) → term 목록. 원문에 있는 것만 남긴다.

**Gate 1**
1. span이 원문에 그대로 있는가 (verbatim 검사)
2. 출력 형식(JSON 스키마) 준수 (pydantic 검증)
3. 이미 있는 term이면 거절하지 말고 기존 인스턴스에 연결 (문서 단위 병합)

참고 설계
- OntoGen: 문장 단위로 용어·약어 추출, 원문에 없는 용어 삭제, self-consistency(여러 번 생성해 일치하는 것만 사용)
- TRACE-KG: 모든 term에 source chunk ID를 남긴다
- Axiom document-entities: 청크별 LLM 후보 → 원문 대조·범주 검사 → 문서 단위 병합

입력: `bpi/documents/bpi2019-purchase-process.txt`

## 설정

In [1]:
import json
import os
import re
from collections import Counter, defaultdict
from pathlib import Path
from typing import Literal

from dotenv import load_dotenv
from google import genai
from google.genai import types
from pydantic import BaseModel, Field, ValidationError

# 노트북은 ontology_extraction/ 에서 실행된다고 가정 (저장소 루트 = 상위 폴더)
PROJECT_ROOT = Path("..")
load_dotenv(PROJECT_ROOT / ".env")

GEMINI_API_KEY = os.environ["GEMINI_API_KEY"]
GEMINI_MODEL = os.environ["GEMINI_MODEL"]

DATASETS_DIR = Path("/Users/soo/code/datasets/")
INPUT_PATH = DATASETS_DIR / "bpi" / "documents" / "bpi2019-purchase-process.txt"

OUTPUT_DIR = Path("outputs") / "p1"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

N_SAMPLES = 3        # self-consistency 샘플 수
MIN_VOTES = 2        # 이 수 이상 샘플에서 나온 term만 채택
TEMPERATURE = 0.7    # 샘플 간 다양성을 위해 0보다 크게

client = genai.Client(api_key=GEMINI_API_KEY)
print(GEMINI_MODEL, INPUT_PATH.exists())

gemini-3.5-flash True


## 문서 읽기와 청크 분할

문서는 `N. TITLE` + `Source: ...` 형태의 섹션으로 되어 있다. 섹션 하나를 청크 하나로 쓰고, 각 청크에 `chunk_id`와 문서 내 시작 offset을 남긴다.

In [2]:
doc_text = INPUT_PATH.read_text(encoding="utf-8")
doc_id = INPUT_PATH.stem

SECTION_RE = re.compile(r"^(\d+)\. [A-Z].*$", re.MULTILINE)

starts = [0] + [m.start() for m in SECTION_RE.finditer(doc_text)]
bounds = list(zip(starts, starts[1:] + [len(doc_text)]))

chunks = []
for i, (s, e) in enumerate(bounds):
    text = doc_text[s:e]
    title = text.strip().splitlines()[0]
    chunks.append({"chunk_id": f"{doc_id}#c{i:02d}", "start": s, "end": e, "title": title, "text": text})

for c in chunks:
    print(c["chunk_id"], c["start"], len(c["text"]), c["title"][:70])

bpi2019-purchase-process#c00 0 1143 BPI CHALLENGE 2019: PURCHASE-TO-PAY PROCESS
bpi2019-purchase-process#c01 1143 761 1. BUSINESS CONTEXT AND PROCESS SCOPE
bpi2019-purchase-process#c02 1904 2200 2. BUSINESS CONCEPTS AND THEIR RELATIONSHIPS
bpi2019-purchase-process#c03 4104 1078 3. PROCESS VARIANT: THREE-WAY MATCHING, INVOICE AFTER GOODS RECEIPT
bpi2019-purchase-process#c04 5182 1108 4. PROCESS VARIANT: THREE-WAY MATCHING, INVOICE BEFORE GOODS RECEIPT
bpi2019-purchase-process#c05 6290 988 5. PROCESS VARIANT: TWO-WAY MATCHING
bpi2019-purchase-process#c06 7278 951 6. PROCESS VARIANT: CONSIGNMENT
bpi2019-purchase-process#c07 8229 758 7. VARIATIONS, CHANGES, AND EXCEPTIONS
bpi2019-purchase-process#c08 8987 1706 8. EXPECTED PROCESS RULES AND THE MEANING OF EVIDENCE
bpi2019-purchase-process#c09 10693 1188 9. RECORDING LIMITATIONS THAT AFFECT PROCESS INTERPRETATION
bpi2019-purchase-process#c10 11881 761 10. PROCESS PERFORMANCE CONCEPTS


## 출력 스키마

term은 문서의 **표면 표현(원문 span)** 이다. 정규화(p2)와 concept 묶기(p3)는 여기서 하지 않는다.
`kind`는 뒤 단계를 위한 힌트일 뿐 타입 결정이 아니다.

In [3]:
TermKind = Literal["thing", "activity", "role", "document", "attribute", "state", "rule", "abbreviation", "identifier", "other"]


class ExtractedTerm(BaseModel):
    span: str = Field(description="Exact surface form copied character-for-character from the text. Do not normalize, singularize, or paraphrase.")
    sentence: str = Field(description="The full sentence from the text that contains the span, copied verbatim.")
    kind: TermKind = Field(description="Coarse hint of what the term denotes.")
    expansion: str | None = Field(default=None, description="If the span is an abbreviation, its expansion as written in the text; otherwise null.")


class ExtractionResult(BaseModel):
    terms: list[ExtractedTerm]

## 프롬프트

In [4]:
SYSTEM_PROMPT = """You extract domain terms from a business process document to seed an ontology.

A term is a surface expression in the text that names a domain thing, activity, role, document, attribute, state, rule, abbreviation, or identifier.

Rules:
- Copy every span EXACTLY as it appears in the text (same casing, punctuation, plural form). Never invent, paraphrase, merge, or normalize.
- Prefer the longest meaningful noun phrase or activity name (e.g. "goods receipt", "Record Invoice Receipt", "purchase order item").
- Include abbreviations as separate terms and give their expansion only if the text states it.
- Skip generic words (e.g. "process", "analysis", "report"), citation metadata (authors, URLs, page numbers), and pure numbers without domain meaning.
- Each distinct span once per chunk; give one supporting sentence copied verbatim.
"""


def build_prompt(chunk: dict) -> str:
    return f"Chunk ID: {chunk['chunk_id']}\n\nText:\n<<<\n{chunk['text']}\n>>>\n\nExtract the terms."

## Gemini 호출 (self-consistency 샘플링)

In [5]:
def extract_once(chunk: dict, seed: int) -> tuple[list[ExtractedTerm], str | None]:
    """한 번 호출. (terms, error) 반환. 스키마 위반이면 error에 사유를 남긴다 (Gate 1-②)."""
    resp = client.models.generate_content(
        model=GEMINI_MODEL,
        contents=build_prompt(chunk),
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT,
            response_mime_type="application/json",
            response_schema=ExtractionResult,
            temperature=TEMPERATURE,
            seed=seed,
        ),
    )
    try:
        return ExtractionResult.model_validate_json(resp.text).terms, None
    except ValidationError as e:
        return [], f"schema: {e.errors()[:3]}"


raw_samples = {}   # chunk_id -> list[list[ExtractedTerm]]
schema_errors = []

for c in chunks:
    samples = []
    for k in range(N_SAMPLES):
        terms, err = extract_once(c, seed=k)
        if err:
            schema_errors.append({"chunk_id": c["chunk_id"], "sample": k, "error": err})
        samples.append(terms)
    raw_samples[c["chunk_id"]] = samples
    print(c["chunk_id"], [len(s) for s in samples])

print("schema errors:", len(schema_errors))

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


bpi2019-purchase-process#c00 [14, 11, 8]


bpi2019-purchase-process#c01 [21, 17, 19]


bpi2019-purchase-process#c02 [35, 40, 46]


bpi2019-purchase-process#c03 [24, 21, 23]


...

bpi2019-purchase-process#c06 [16, 16, 19]


bpi2019-purchase-process#c07 [17, 16, 16]


bpi2019-purchase-process#c08 [14, 21, 15]


bpi2019-purchase-process#c09 [13, 14, 15]


bpi2019-purchase-process#c10 [13, 14, 16]
schema errors: 0


## Gate 1

- ① **verbatim**: span과 sentence가 청크 원문에 그대로 있는가. 없으면 기각. 대소문자만 다른 경우는 원문 표기로 교정하고 `case_fixed` 표시.
- **self-consistency**: `N_SAMPLES` 중 `MIN_VOTES` 이상에서 나온 span만 채택.
- ③ **병합**: 같은 span은 하나의 term 인스턴스로 묶고, 청크별 mention(offset 포함)을 연결한다.

In [6]:
def locate(span: str, chunk: dict) -> tuple[str | None, bool]:
    """청크에서 span을 찾아 (원문 표기, case_fixed) 반환. 못 찾으면 (None, False)."""
    if span in chunk["text"]:
        return span, False
    m = re.search(re.escape(span), chunk["text"], flags=re.IGNORECASE)
    if m:
        return m.group(0), True
    return None, False


accepted_mentions = []   # Gate 통과한 (chunk, span) 단위
rejected = []

for c in chunks:
    votes = Counter()
    first_seen = {}
    for k, sample in enumerate(raw_samples[c["chunk_id"]]):
        seen_in_sample = set()
        for t in sample:
            surface, case_fixed = locate(t.span.strip(), c)
            if surface is None:
                rejected.append({"chunk_id": c["chunk_id"], "sample": k, "span": t.span, "reason": "span_not_in_source"})
                continue
            if surface in seen_in_sample:
                continue
            seen_in_sample.add(surface)
            votes[surface] += 1
            first_seen.setdefault(surface, (t, case_fixed))

    for surface, n in votes.items():
        t, case_fixed = first_seen[surface]
        if n < MIN_VOTES:
            rejected.append({"chunk_id": c["chunk_id"], "span": surface, "votes": n, "reason": "low_consistency"})
            continue
        offsets = [c["start"] + m.start() for m in re.finditer(re.escape(surface), c["text"])]
        accepted_mentions.append({
            "span": surface,
            "chunk_id": c["chunk_id"],
            "votes": n,
            "kind": t.kind,
            "expansion": t.expansion,
            "sentence": t.sentence,
            "sentence_verbatim": t.sentence.strip() in c["text"],
            "case_fixed": case_fixed,
            "offsets": offsets,
        })

print("accepted mentions:", len(accepted_mentions))
print("rejected:", Counter(r["reason"] for r in rejected))

accepted mentions: 210
rejected: Counter({'low_consistency': 35})


In [7]:
# ③ 문서 단위 병합: 같은 span -> 하나의 term 인스턴스, mention 연결
merged = defaultdict(lambda: {"mentions": [], "kinds": Counter(), "expansions": set()})
for m in accepted_mentions:
    e = merged[m["span"]]
    e["mentions"].append({k: m[k] for k in ("chunk_id", "votes", "offsets", "sentence", "sentence_verbatim", "case_fixed")})
    e["kinds"][m["kind"]] += 1
    if m["expansion"]:
        e["expansions"].add(m["expansion"])

terms = []
for i, (span, e) in enumerate(sorted(merged.items(), key=lambda kv: kv[0].lower())):
    terms.append({
        "term_id": f"{doc_id}#t{i:03d}",
        "span": span,
        "kind": e["kinds"].most_common(1)[0][0],
        "expansion": sorted(e["expansions"]) or None,
        "n_chunks": len(e["mentions"]),
        "mentions": e["mentions"],
    })

print("terms:", len(terms))
print(Counter(t["kind"] for t in terms))

terms: 162
Counter({'activity': 55, 'document': 24, 'thing': 23, 'attribute': 16, 'rule': 12, 'other': 12, 'role': 10, 'state': 4, 'identifier': 4, 'abbreviation': 2})


## 결과 확인과 저장

In [8]:
for t in sorted(terms, key=lambda t: -t["n_chunks"])[:40]:
    exp = f" = {t['expansion']}" if t["expansion"] else ""
    print(f"{t['n_chunks']:>2}  {t['kind']:<12} {t['span']}{exp}")

 6  activity     invoice clearance
 5  activity     goods receipt
 5  activity     invoice receipt
 4  thing        purchase order item
 4  activity     purchase order item creation
 3  document     event log
 3  document     invoice
 3  activity     invoice receipt recording
 3  document     purchase order document
 3  abbreviation SRM
 2  thing        Consignment
 2  abbreviation EC
 2  document     EC purchase orders
 2  document     framework orders
 2  activity     Goods receipt
 2  activity     goods receipt recording
 2  activity     invoice creation
 2  attribute    invoice value
 2  thing        item
 2  other        item cases
 2  activity     item creation
 2  document     order confirmation
 2  state        payment block
 2  document     purchase order
 2  thing        purchase order items
 2  activity     purchase requisition creation
 2  activity     quantity changes
 2  document     recurring invoices
 2  activity     requisition
 1  attribute    active work duration
 1 

In [9]:
run_meta = {
    "doc_id": doc_id,
    "input": str(INPUT_PATH),
    "model": GEMINI_MODEL,
    "n_samples": N_SAMPLES,
    "min_votes": MIN_VOTES,
    "temperature": TEMPERATURE,
    "n_chunks": len(chunks),
    "n_terms": len(terms),
    "n_rejected": len(rejected),
    "n_schema_errors": len(schema_errors),
}

(OUTPUT_DIR / "terms.json").write_text(json.dumps({"meta": run_meta, "terms": terms}, ensure_ascii=False, indent=2), encoding="utf-8")
(OUTPUT_DIR / "rejected.json").write_text(json.dumps({"rejected": rejected, "schema_errors": schema_errors}, ensure_ascii=False, indent=2), encoding="utf-8")
(OUTPUT_DIR / "raw_samples.json").write_text(
    json.dumps({cid: [[t.model_dump() for t in s] for s in samples] for cid, samples in raw_samples.items()}, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
run_meta

{'doc_id': 'bpi2019-purchase-process',
 'input': '/Users/soo/code/datasets/bpi/documents/bpi2019-purchase-process.txt',
 'model': 'gemini-3.5-flash',
 'n_samples': 3,
 'min_votes': 2,
 'temperature': 0.7,
 'n_chunks': 11,
 'n_terms': 162,
 'n_rejected': 35,
 'n_schema_errors': 0}